# Adım 3: Spark + Delta Lake — Bronze / Silver / Gold
**Kişi 2 sorumluluğu** — `feature/spark-eda` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month
import os

DATA_PATH   = './data/archive/daily_weather.parquet'
BRONZE_PATH = './delta_lake/bronze'
SILVER_PATH = './delta_lake/silver'
GOLD_PATH   = './delta_lake/gold'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateDataPipeline')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .config('spark.sql.parquet.datetimeRebaseModeInRead', 'CORRECTED')
        .config('spark.sql.parquet.int96RebaseModeInRead', 'CORRECTED')
        .getOrCreate()
    )

def write_bronze(spark):
    print('[BRONZE] Ham veri okunuyor...')
    df = spark.read.parquet(DATA_PATH)
    count = df.count()
    print(f'[BRONZE] Kayit sayisi: {count:,}')
    df.write.format('delta').mode('overwrite').save(BRONZE_PATH)
    print(f'[BRONZE] Yazildi -> {BRONZE_PATH}')
    return df

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
write_bronze(spark)